# Exchange Rates Exploratory Analysis

This notebook reads the Iceberg tables produced by the pipeline through the MinIO-backed Spark catalog. Run the pipeline first, then execute the cells from top to bottom.

In [ ]:
import sys
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

sys.path.insert(0, '/app')
from src.config import load_config
from src.spark_session import build_spark_session

config = load_config()
spark = build_spark_session(config)
spark.conf.set('spark.sql.repl.eagerEval.enabled', 'true')
CATALOG = f'{config.iceberg_catalog_name}.{config.iceberg_database_name}'
print(f'Catalog: {CATALOG}')
print(f'Spark: {spark.version}')

In [ ]:
tables = spark.sql(f'SHOW TABLES IN {CATALOG}')
tables.select('tableName').show(truncate=False)

## Exchange-rate coverage and distribution

In [ ]:
rates = spark.table(f'{CATALOG}.fact_exchange_rates')
rates.printSchema()
rates.select('rate_date', 'base_currency', 'quote_currency', 'exchange_rate', 'daily_variation_pct').show(10, truncate=False)

currency_summary = (
    rates.groupBy('quote_currency')
    .agg(
        F.count('*').alias('observations'),
        F.avg('exchange_rate').alias('average_rate'),
        F.min('exchange_rate').alias('minimum_rate'),
        F.max('exchange_rate').alias('maximum_rate'),
        F.stddev('exchange_rate').alias('rate_stddev'),
    )
    .orderBy('quote_currency')
)
currency_summary.show(truncate=False)

In [ ]:
daily_coverage = (
    rates.groupBy('rate_date')
    .agg(F.countDistinct('quote_currency').alias('currencies_present'))
    .orderBy('rate_date')
)
daily_coverage.show(15)

coverage_pd = daily_coverage.toPandas()
coverage_pd.plot(x='rate_date', y='currencies_present', figsize=(12, 4), legend=False, title='Daily currency coverage')
plt.ylabel('Currencies with observations')
plt.xlabel('Date')
plt.tight_layout()
plt.show()

## Monthly metrics and anomalies

In [ ]:
monthly = spark.table(f'{CATALOG}.monthly_exchange_rate_metrics')
monthly.orderBy('quote_currency', 'calendar_year', 'calendar_month').show(20, truncate=False)

anomalies = spark.table(f'{CATALOG}.exchange_rate_anomalies')
print(f'Anomaly rows: {anomalies.count()}')
anomalies.orderBy('rate_date').show(20, truncate=False)

In [ ]:
quality = spark.table(f'{CATALOG}.data_quality_report')
quality.show(30, truncate=False)

# Stop the notebook Spark session when analysis is complete.
spark.stop()